In [2]:

import pandas as pd
import numpy as np
import re
import nltk
import string
import nltk
from nltk.corpus import stopwords
import statistics
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,SimpleRNN,GRU,Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [3]:
df = pd.read_csv('IMDB Dataset.txt',nrows=25000)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91976\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
def remove_html_tags(text):
  clean_text = re.sub(r'<.*?>', '', text)
  return clean_text

df['review'] = df['review'].apply(remove_html_tags)
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
import string
punc = string.punctuation
punc

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [8]:
def remove_punc(text):
  clean_text = "".join([char for char in text if char not in punc])
  return clean_text

df['review'] = df['review'].apply(remove_punc)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production The filming tech...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically theres a family where a little boy J...,negative
4,Petter Matteis Love in the Time of Money is a ...,positive


In [9]:
df['review'] =  df['review'].str.lower()

In [10]:


stop_words = set(stopwords.words('english'))

def stp_words(text):
    return [word for word in str(text).split() if word.lower() not in stop_words]

df['review'] = df['review'].apply(stp_words)

In [11]:

token = Tokenizer()
token.fit_on_texts(df['review'])

In [12]:
token = Tokenizer()
token.fit_on_texts(df['review'])

In [13]:
len(token.word_index)
     

144060

In [14]:

seq = token.texts_to_sequences(df['review'])

In [15]:

average = statistics.mean

In [16]:
average([len(x) for x in seq])

119.99108

In [17]:
padding = pad_sequences(seq,maxlen=220,padding='pre')

In [18]:
padding.shape
     

(25000, 220)

In [19]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

In [20]:

x= padding
y = df['sentiment']

In [21]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.175, random_state=42)

In [22]:
x_train.shape , x_test.shape , y_train.shape , y_test.shape

((20625, 220), (4375, 220), (20625,), (4375,))

In [23]:
y.value_counts()

sentiment
0    12526
1    12474
Name: count, dtype: int64

In [24]:
vocab_size = len(token.word_index) + 1
print(f"Vocabulary size in x_train: {vocab_size}")

Vocabulary size in x_train: 144061


In [25]:
from tensorflow.keras.layers import Input
m1 = Sequential()
m1.add(Input(shape=(220,)))
m1.add(Embedding(vocab_size, 100))
m1.add(SimpleRNN(150, return_sequences=True))
m1.add(SimpleRNN(50, return_sequences=True))
m1.add(SimpleRNN(25))
m1.add(Dense(1, activation='sigmoid'))

In [26]:
m1.compile(optimizer='adam',
           loss='binary_crossentropy',
           metrics=['accuracy'])
m1.fit(x_train,y_train,epochs=2,batch_size=500,validation_data=(x_test,y_test))


Epoch 1/2
42/42 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - accuracy: 0.5065 - loss: 0.6986 - val_accuracy: 0.4917 - val_loss: 0.6930
Epoch 2/2
42/42 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.6469 - loss: 0.6371 - val_accuracy: 0.5211 - val_loss: 0.7400


In [27]:
m2 = Sequential()
m2.add(Input(shape=(220,)))
m2.add(Embedding(144771,output_dim=250,input_length=220))
m2.add(Bidirectional(LSTM(150,return_sequences=True)))
m2.add(Bidirectional(GRU(100,return_sequences=True)))
m2.add(Dropout(0.2))
m2.add(Bidirectional(LSTM(50,return_sequences=True)))
m2.add(Bidirectional(GRU(30,return_sequences=True)))
m2.add(Dropout(0.2))
m2.add(LSTM(10))
m2.add(Dense(1,activation='sigmoid'))

m2.summary()

c:\Users\91976\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 220, 250)       │    36,192,750 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 220, 300)       │       481,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 220, 200)       │       241,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 220, 200)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 220, 100)       │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 220, 60)        │        23,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 220, 60)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 10)             │         2,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,042,161 (141.30 MB)

 Trainable params: 37,042,161 (141.30 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
m2.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
m2.fit(x,y,epochs=1,batch_size=500)

50/50 ━━━━━━━━━━━━━━━━━━━━ 622s 12s/step - accuracy: 0.7872 - loss: 0.4507


In [31]:
text = ["""An unexpectedly brilliant film with powerful performances, sharp writing, and emotional depth. Every scene was engaging, the music was memorable, and the ending was deeply satisfying. Easily one of the best movies I have watched this year."""]

seq = token.texts_to_sequences(text)
padding = pad_sequences(seq, maxlen=220, padding='pre')

pred = float(m2.predict(padding)[0][0])

if pred < 0.2:
    print("flop movie")
elif pred < 0.55:
    print("average movie")
elif pred < 0.8:
    print("good movie")
else:
    print("Blockbuster movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Blockbuster movie
